<a href="https://colab.research.google.com/github/Shameen-ghyas/Machine-Learning/blob/main/Arabic_models_using_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shameen Ghyas CS23002
# Nabeeha Ashar CS23005
# Afra Khurram Ansari CS23008
# Zunaira Zainab Ansari CS23026

# Arabic Character Recognition with YOLOv8 and Gradio

In [ ]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.1 MB/s eta 0:00:00


In [ ]:
#imports
from google.colab import userdata
from huggingface_hub import hf_hub_download
import zipfile
from ultralytics import YOLO
import json
import yaml
import gradio as gr
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Setup and Data Loading
This section handles the initial setup, including loading the dataset from Hugging Face Hub.

In [ ]:
# Load the dataset as a zip file from Hugging Face Hub
# and extract its contents to the /content/arabic_dataset directory.
path = hf_hub_download(
    repo_id="Shameen-ghyas/arabic-character-dataset",
    filename="arabic_dataset.zip",
    repo_type="dataset"
)
with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall('/content/arabic_dataset')

# Define the path to the YAML configuration file for the dataset.
yaml_path = '/content/arabic_dataset/data.yaml'
print("Dataset ready!")

arabic_dataset.zip:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

Dataset ready!


## Model Training

# Model 1

In [ ]:
# from ultralytics import YOLO

# model1 = YOLO('yolov8n.pt')

# model1.train(
#     data=yaml_path,
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     lr0=0.01,
#     dropout=0.0,
#     optimizer='SGD',
#     patience=10,
#     project='arabic_runs',
#     name='model1_sgd',
#     exist_ok=True
# )

# Model 2


In [ ]:
# model2 = YOLO('yolov8n.pt')
# model2.train(
#     data=yaml_path,
#     epochs=50,
#     imgsz=640,
#     batch=32,
#     lr0=0.005,
#     momentum=0.9,
#     weight_decay=0.001,
#     dropout=0.1,
#     optimizer='SGD',
#     patience=10,
#     project='arabic_runs',
#     name='model2_sgd',
#     exist_ok=True
# )

# Model 3

In [ ]:
# from ultralytics import YOLO

# model3 = YOLO('yolov8n.pt')

# model3.train(
#     data=yaml_path,
#     epochs=50,      # back to 50
#     imgsz=640,
#     batch=32,
#     lr0=0.001,
#     dropout=0.3,
#     optimizer='AdamW',
#     patience=10,
#     project='arabic_runs',
#     name='model3_adam',
#     exist_ok=True,
#     weight_decay= 0.001
# )

# Model Uploading to HF
Uploading (saving) the trained models to HF for it to be accessed easily.

---


Note: The same code can be used for all three models, only the path variables need to be changed.

In [ ]:
# from huggingface_hub import HfApi

# api = HfApi()

# api.upload_file(
#     path_or_fileobj='/kaggle/working/arabic_runs/model1_sgd/weights/best.pt',
#     path_in_repo='model1_sgd_best.pt',
#     repo_id="Shameen-ghyas/arabic-character-dataset",
#     repo_type="dataset",
#     token=HF_TOKEN
# )

# print('Model 1 saved to HF!')

## Model Loading
Here, we load the pre-trained YOLOv8 models from Hugging Face Hub. Three different models, trained with various optimizers, are loaded for comparison or selection.

In [ ]:
# Define a dictionary mapping descriptive names to the filenames of the pre-trained models.
models_info = {
    'Model 1 (SGD)' : 'model1_sgd_best.pt',
    'Model 2 (SGD)': 'model2_sgd_best.pt',
    'Model 3 (AdamW)': 'model3_adam_best.pt',
}

loaded = {}
# Iterate through each model defined in `models_info`.
for name, filename in models_info.items():
    # Download each model file from the Hugging Face Hub.
    p = hf_hub_download(
        repo_id="Shameen-ghyas/arabic-character-dataset",
        filename=filename,
        repo_type="dataset"

    )
    # Initialize a YOLO model with the downloaded weights and store it.
    loaded[name] = YOLO(p)
    print(f"{name} loaded!")

# Assign each loaded model to a specific variable for easy access.
m1 = loaded['Model 1 (SGD)']
m2 = loaded['Model 2 (SGD)']
m3 = loaded['Model 3 (AdamW)']

model1_sgd_best.pt:   0%|          | 0.00/6.26M [00:00<?, ?B/s]

Model 1 (SGD) loaded!


model2_sgd_best.pt:   0%|          | 0.00/6.26M [00:00<?, ?B/s]

Model 2 (SGD) loaded!


model3_adam_best.pt:   0%|          | 0.00/6.26M [00:00<?, ?B/s]

Model 3 (AdamW) loaded!


In [ ]:
# Download the JSON file containing all model results from Hugging Face Hub.
path = hf_hub_download(
    repo_id="Shameen-ghyas/arabic-character-dataset",
    filename="results_all.json",
    repo_type="dataset"
)

# Open and load the JSON data into the `results_all` dictionary.
with open(path, "r") as f:
    results_all = json.load(f)

results_all.json: 0.00B [00:00, ?B/s]

## Evaluation Results
This section loads and displays the performance metrics (mAP, Precision, Recall) for each model on the validation, test, and training splits.

In [ ]:
print(f"\n{'='*65}")
print(f"{'Model':<18} {'Split':<8} {'mAP50':>7} {'mAP50-95':>10} {'Precision':>10} {'Recall':>8}")
print(f"{'='*65}")

for model_name, splits in results_all.items():
    for split, metrics in splits.items():
        print(f"{model_name:<18} {split:<8} "
              f"{metrics['mAP50']:.4f} "
              f"{metrics['mAP50-95']:.4f} "
              f"{metrics['Precision']:.4f} "
              f"{metrics['Recall']:.4f}")
    print(f"{'-'*65}")


Model              Split      mAP50   mAP50-95  Precision   Recall
Model 1 (SGD)      val      0.9215 0.7874 0.8948 0.8588
Model 1 (SGD)      test     0.9190 0.7801 0.8909 0.8677
Model 1 (SGD)      train    0.9938 0.9741 0.9993 0.9962
-----------------------------------------------------------------
Model 2 (SGD)      val      0.9194 0.7811 0.9193 0.8408
Model 2 (SGD)      test     0.9147 0.7809 0.8871 0.8716
Model 2 (SGD)      train    0.9939 0.9817 0.9994 0.9970
-----------------------------------------------------------------
Model 3 (AdamW)    val      0.9201 0.7812 0.8973 0.8590
Model 3 (AdamW)    test     0.9239 0.7734 0.8896 0.8483
Model 3 (AdamW)    train    0.9939 0.9627 0.9991 0.9958
-----------------------------------------------------------------


## Class Mapping
Here, we load the class names from the dataset's YAML file and create mappings between class IDs, Arabic characters, and their English transliterations. This is essential for interpreting model predictions.

In [ ]:
# Load the YAML file to get the ordered list of class names (character IDs).
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)
yaml_order = data['names']

# Define a mapping from numeric IDs to Arabic characters.
arabic_map = {
    '0':'ا','1':'ب','2':'ت','3':'ث','4':'ج','5':'ح','6':'خ','7':'د','8':'ذ','9':'ر',
    '10':'ز','11':'س','12':'ش','13':'ص','14':'ض','15':'ط','16':'ظ','17':'ع','18':'غ','19':'ف',
    '20':'ق','21':'ك','22':'ل','23':'م','24':'ن','26':'ه','25':'و','27':'ي'
}
# Define a mapping from numeric IDs to English transliterations of Arabic characters.
english_map = {
    '0':'Alef','1':'Baa','2':'Ta','3':'saa','4':'Jeem','5':'Ha','6':'Kha','7':'Dal','8':'Zal','9':'Ra',
    '10':'Zaa','11':'Seen','12':'Sheen','13':'Suad','14':'Duad','15':'Tau','16':'Zhau','17':'Ain','18':'Ghain','19':'Fa',
    '20':'Qauf','21':'Kaf','22':'Lam','23':'Meem','24':'Nun','26':'Ha','25':'Waw','27':'Ya'
}
# Create ordered lists of Arabic and English names based on `yaml_order`.
ARABIC_NAMES  = [arabic_map[k] for k in yaml_order]
ENGLISH_NAMES = [english_map[k] for k in yaml_order]
print("Class mapping ready!")

Class mapping ready!


# Final Selection
Model 1 was selected as the final model because it achieved the best overall performance among all three trained models. It produced the highest detection precision and more consistent predictions on unseen images, especially for visually similar Arabic characters. In addition, the SGD optimizer provided better generalization and more stable results compared to the other configurations, making Model 1 the most reliable choice for the final inference interface.


## Gradio Interface for Prediction
This section defines a `predict` function that uses YOLOv8 Model1 to detect Arabic characters in an uploaded image. A Gradio interface is then created to allow interactive testing of the model.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

def predict(image):
    if image is None:
        return None, "No image has been uploaded!"

    preds = m1.predict(source=image, conf=0.25, verbose=False)
    boxes = preds[0].boxes

    annotated_img = preds[0].plot()
    annotated_img = annotated_img[:, :, ::-1]

    if len(boxes) == 0:
        return Image.fromarray(annotated_img), "No character detected. Try a clearer image."

    output = f"{ '='*40 }\n"
    output += f"  {len(boxes)} character(s) detected\n"
    output += f"{ '='*40 }\n\n"

    for i, box in enumerate(boxes):
        cls_id  = int(box.cls[0])
        conf    = float(box.conf[0])
        arabic  = ARABIC_NAMES[cls_id]
        english = ENGLISH_NAMES[cls_id]
        pct     = int(conf * 100)
        bar     = '█' * (pct // 5) + '░' * (20 - pct // 5)
        output += f"[{i+1}] Arabic  : {arabic}\n"
        output += f"     English : {english}\n"
        output += f"     Conf    : {bar} {conf:.2f}\n\n"

    return Image.fromarray(annotated_img), output

gr.Interface(
    fn=predict,
    inputs=gr.Image(
        type="numpy",
        # label="Upload Arabic Character Image",
        sources=["upload"]
    ),
    outputs=[
        gr.Image(label="Detection Output"),
        gr.Textbox(label="Detection Results", lines=12)  # text results
    ],
    title="Arabic Character Recognizer",
    description="Model 1 (SGD): Upload an image to detect Arabic characters",
    theme=gr.themes.Soft(),
    flagging_mode="never"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdb463b6cc024340f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
